# 10. Lexical Term Accuracy & Rare-Word Precision Study
Analyzes the master lexical corpus itself, then demonstrates the rare-word / terminology evaluation machinery on synthetic example translations (real model predictions come from notebook 09's saved output on the actual training/inference environment).

In [ ]:
# ============================================================
# PATH & REPO AUTO-SYNC BOOSTER — Guarantees latest project code
# ============================================================
import os, sys, site, urllib.request, zipfile

# Purge cached 'src' modules from memory so updated files take effect immediately
for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)

try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)

home       = os.path.expanduser('~')
proj_dir   = os.path.join(home, 'Ekegusii-LLM-Translation-main')
sync_tag   = os.path.join(proj_dir, 'configs', 'models', 'v2_mistral_earlystop_v5.tag')

# Auto-sync if folder is missing OR outdated (lacks v2_mistral_earlystop_v5.tag)
if not os.path.isfile(sync_tag):
    print('🔄 Outdated or missing repository detected. Auto-syncing latest code from GitHub...')
    zip_path = os.path.join(home, 'repo.zip')
    urllib.request.urlretrieve('https://github.com/aykahsay/Ekegusii-LLM-Translation/archive/refs/heads/main.zip', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(home)
    os.remove(zip_path)
    print('✅ Repository auto-synced to latest main commit!')

if os.path.isdir(proj_dir):
    os.chdir(proj_dir)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working Directory : {os.getcwd()}')
print(f'Python Kernel     : {sys.executable}')


In [2]:
# ============================================================
# ABI CHECK -- numpy/pandas binary compatibility.
# Some Jupyter hosts (e.g. Kineses Cloud conda envs) ship a numpy/pandas
# pair whose compiled C-extension ABI doesn't match, raising
# "numpy.dtype size changed, may indicate binary incompatibility" the
# moment pandas -- and therefore anything importing it, like
# src.master_corpus -- is loaded. Detect and fix it BEFORE any pandas
# import below (see notebooks/00_setup_environment.ipynb for the
# original version of this check).
# ============================================================
import subprocess
import sys


def _abi_ok():
    try:
        import numpy  # noqa: F401
        import pandas  # noqa: F401
        return True
    except ValueError as exc:
        if "binary incompatibility" in str(exc):
            return False
        raise


if not _abi_ok():
    print("numpy/pandas ABI mismatch detected -- attempting fix...")
    fix_a = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "numpy>=2.0.0"],
        capture_output=True, text=True,
    )
    if fix_a.returncode != 0:
        print("  numpy upgrade failed (read-only env?) -- downgrading pandas instead...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "pandas==2.2.3"],
            capture_output=True, text=True,
        )
    raise RuntimeError(
        "Fixed numpy/pandas ABI mismatch via pip -- you MUST restart the kernel now "
        "(Kernel -> Restart Kernel) and re-run this notebook from the top. The fix "
        "cannot take effect in the current running process."
    )
else:
    print("numpy/pandas ABI OK.")


numpy/pandas ABI OK.


In [3]:
from src.master_corpus.manager import MasterCorpusManager

manager = MasterCorpusManager()
lexical_df = manager.load_lexical_corpus()
print(f'{len(lexical_df)} lexical entries')
lexical_df.head()

INFO | Loaded Master Lexical Corpus: 268 terms.


268 lexical entries


,lexicon_id,English,Kiswahili,Ekegusii,source,dataset_origin
0,1,NaN,jambo,amangʼana,Dictionary,Online_Glosbe_Swahili_Ekegusii_Dictionary.csv
1,2,NaN,rafiki,omosani,Dictionary,Online_Glosbe_Swahili_Ekegusii_Dictionary.csv
2,3,NaN,mtu,omonto,Dictionary,Online_Glosbe_Swahili_Ekegusii_Dictionary.csv
3,4,NaN,watu,"abanto, abantu",Dictionary,Online_Glosbe_Swahili_Ekegusii_Dictionary.csv
4,5,NaN,mume,omosacha,Dictionary,Online_Glosbe_Swahili_Ekegusii_Dictionary.csv


## Column coverage
See `docs/datasets.md`: English is currently 0% populated.

In [4]:
for lang in ['English', 'Kiswahili', 'Ekegusii']:
    non_null = lexical_df[lang].notna().sum()
    print(f'{lang}: {non_null}/{len(lexical_df)} ({100*non_null/len(lexical_df):.1f}%)')

English: 0/268 (0.0%)
Kiswahili: 268/268 (100.0%)
Ekegusii: 268/268 (100.0%)


## Rare-word identification on the sentence corpus

In [5]:
from src.tokenizer.rare_words import RareWordIdentifier

sentence_df = manager.load_sentence_corpus().sample(2000, random_state=42)
eke_sentences = sentence_df['Ekegusii'].dropna().astype(str).tolist()

identifier = RareWordIdentifier()
rare_words = identifier.identify_rare_words(eke_sentences, max_frequency=2)
print(f'{len(rare_words)} rare Ekegusii word types (freq <= 2) in this sample')
rare_words[:20]

C:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO | Loaded Master Sentence Corpus: 49,277 concepts.


INFO | Identified 7,967 rare word type(s) (freq <= 2).


7967 rare Ekegusii word types (freq <= 2) in this sample


['000gosimekwa',
 '100',
 '10th',
 '110',
 '112',
 '116',
 '1200',
 '121',
 '127',
 '128',
 '12na',
 '133',
 '13na',
 '1400',
 '146',
 '147',
 '14na',
 '150',
 '15na',
 '16na']

## Terminology consistency checker (demo)

In [6]:
from src.evaluation.terminology import TerminologyConsistencyChecker

checker = TerminologyConsistencyChecker.build_from_lexical_corpus(
    lexical_df, source_lang='Kiswahili', target_lang='Ekegusii', min_term_length=4
)
print(f'{len(checker.terminology_map)} curated terms loaded from the lexical corpus')

124 curated terms loaded from the lexical corpus
